In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
from google.cloud import storage


In [5]:
project_id = "project-19af6ebc-311f-448b-851"
bucket = "fundamentos_sobre_cloud"
file_name = "BR.json"

In [6]:
cliente_gcs = storage.Client(project=project_id)
bucket = cliente_gcs.get_bucket(bucket)
blob = bucket.blob(file_name)

In [8]:
file_content_str = blob.download_as_text()

print(f"Conteúdo do arquivo {file_name}")
print(file_content_str)

Conteúdo do arquivo BR.json
[{"date":"2016-01-01","localName":"Confraternização Universal","name":"New Year's Day","countryCode":"BR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2016-02-08","localName":"Carnaval","name":"Carnival","countryCode":"BR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Bank","Optional"]},{"date":"2016-02-09","localName":"Carnaval","name":"Carnival","countryCode":"BR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Bank","Optional"]},{"date":"2016-03-25","localName":"Sexta-feira Santa","name":"Good Friday","countryCode":"BR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2016-03-27","localName":"Domingo de Páscoa","name":"Easter Sunday","countryCode":"BR","fixed":false,"global":true,"counties":null,"launchYear":null,"types":["Public"]},{"date":"2016-04-21","localName":"Dia de Tiradentes","name":"Tiradentes","countryCode":"

In [10]:
import json

dados_feriados = json.loads(file_content_str)
dados_feriados

[{'date': '2016-01-01',
  'localName': 'Confraternização Universal',
  'name': "New Year's Day",
  'countryCode': 'BR',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public']},
 {'date': '2016-02-08',
  'localName': 'Carnaval',
  'name': 'Carnival',
  'countryCode': 'BR',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Bank', 'Optional']},
 {'date': '2016-02-09',
  'localName': 'Carnaval',
  'name': 'Carnival',
  'countryCode': 'BR',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Bank', 'Optional']},
 {'date': '2016-03-25',
  'localName': 'Sexta-feira Santa',
  'name': 'Good Friday',
  'countryCode': 'BR',
  'fixed': False,
  'global': True,
  'counties': None,
  'launchYear': None,
  'types': ['Public']},
 {'date': '2016-03-27',
  'localName': 'Domingo de Páscoa',
  'name': 'Easter Sunday',
  'countryCode': 'BR',
  'fixed': False,
  'global': True,
  'cou

In [11]:
from google.cloud import bigquery

In [12]:
cliente_bq = bigquery.Client(project=project_id)

In [19]:
consulta_pedidos = """
SELECT order_id, order_status, order_purchase_timestamp, order_estimated_delivery_date, order_delivered_customer_date
FROM `project-19af6ebc-311f-448b-851.olist_dataset.orders`
"""

In [20]:
query_job = cliente_bq.query(consulta_pedidos)
pedidos = query_job.to_dataframe()
pedidos

,order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date
0,a2e4c44360b4a57bdff22f3a4630c173,approved,2017-02-06 20:18:17+00:00,2017-03-01 00:00:00+00:00,NaT
1,132f1e724165a07f6362532bfb97486e,approved,2017-04-25 01:25:34+00:00,2017-05-22 00:00:00+00:00,NaT
2,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19+00:00,2016-09-30 00:00:00+00:00,NaT
3,e5215415bb6f76fe3b7cb68103a0d1c0,canceled,2016-10-22 08:25:27+00:00,2016-10-24 00:00:00+00:00,NaT
4,71303d7e93b399f5bcd537d124c0bcfa,canceled,2016-10-02 22:07:52+00:00,2016-10-25 00:00:00+00:00,NaT
...,...,...,...,...,...
99436,4cccc0d35e7c7a0dc766ad3c4043e33e,unavailable,2018-08-10 09:32:32+00:00,2018-08-15 00:00:00+00:00,NaT
99437,897b4da63b6edde1a33a9fb7caf1dd10,unavailable,2018-07-30 07:38:21+00:00,2018-08-16 00:00:00+00:00,NaT
99438,93881917b8e0f2bf11eec7abbbfe43ec,unavailable,2018-08-11 21:38:00+00:00,2018-08-21 00:00:00+00:00,NaT
99439,4bd0d8aa4756f78245bd56015d4ddcc0,unavailable,2018-08-11 11:56:24+00:00,2018-08-27 00:00:00+00:00,NaT


In [22]:
consulta_atrasos = """
SELECT order_id, order_estimated_delivery_date,
order_delivered_customer_date,
DATE_DIFF(order_delivered_customer_date, order_estimated_delivery_date, DAY) AS atraso_medio_dias
FROM `project-19af6ebc-311f-448b-851.olist_dataset.orders`
WHERE
order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL
AND order_delivered_customer_date > order_estimated_delivery_date
ORDER BY atraso_medio_dias DESC
"""

results = cliente_bq.query(consulta_atrasos)
df_atrasos = results.to_dataframe()
df_atrasos


,order_id,order_estimated_delivery_date,order_delivered_customer_date,atraso_medio_dias
0,1b3190b2dfa9d789e1f14c05b647a14a,2018-03-15 00:00:00+00:00,2018-09-19 23:24:07+00:00,188
1,ca07593549f1816d26a572e06dc1eab6,2017-03-22 00:00:00+00:00,2017-09-19 14:36:39+00:00,181
2,47b40429ed8cce3aee9199792275433f,2018-01-19 00:00:00+00:00,2018-07-13 20:51:31+00:00,175
3,2fe324febf907e3ea3f2aa9650869fa5,2017-04-05 00:00:00+00:00,2017-09-19 17:00:07+00:00,167
4,285ab9426d6982034523a855f55a885e,2017-04-06 00:00:00+00:00,2017-09-19 14:00:04+00:00,166
...,...,...,...,...
7822,eb2e56877e1f053dedc857408122a7b4,2018-08-30 00:00:00+00:00,2018-08-30 01:20:34+00:00,0
7823,124223dc899eca6ac54187a54f1e3632,2018-08-30 00:00:00+00:00,2018-08-30 21:51:12+00:00,0
7824,823e4c9d908f5ec1cad58e1e3e7d716d,2018-08-31 00:00:00+00:00,2018-08-31 02:21:48+00:00,0
7825,d779224365f07d6953c45ef342c824e6,2018-08-31 00:00:00+00:00,2018-08-31 00:48:33+00:00,0


In [26]:
from google.cloud.bigquery.job import WriteDisposition
caminho = "olist_dataset.atrasos"

schema = [
    bigquery.SchemaField("order_id", "STRING"),
    bigquery.SchemaField("order_estimated_delivery_date", "TIMESTAMP"),
    bigquery.SchemaField("order_delivered_customer_date", "TIMESTAMP"),
    bigquery.SchemaField("atraso_medio_dias", "INTEGER"),
]

job_config = bigquery.LoadJobConfig(schema=schema, write_disposition = "WRITE_APPEND")

job = cliente_bq.load_table_from_dataframe(df_atrasos, caminho, job_config=job_config)
job.result()

LoadJob<project=project-19af6ebc-311f-448b-851, location=US, id=f4135d7d-e9c2-427f-a5f1-4fb633a28d40>